## 1. Import Library & Manual Data Splitting

In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# LOAD DATA
df = pd.read_csv('../data/dataset_encoded.csv')

X = df.drop(columns=['Target'])
y = df['Target']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# Pisahkan sisanya menjadi Train (70% dari total) dan Validation (15% dari total)
# Proporsi = 15 / 85 = 0.1765
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# SCALING DATA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Jumlah Data Train: {X_train.shape[0]}")
print(f"Jumlah Data Validation: {X_val.shape[0]}")
print(f"Jumlah Data Test: {X_test.shape[0]}")
print("Data siap masuk pipeline model!")

Jumlah Data Train: 4637
Jumlah Data Validation: 995
Jumlah Data Test: 995
Data siap masuk pipeline model!


## 2. Training Base Model & Evaluasi Validation

In [3]:
# TRAINING BASE MODEL RANDOM FOREST
print("Training Base Model Random Forest...")
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_base.fit(X_train_scaled, y_train)

# Evaluasi ke Validation Set untuk cek performa awal
y_pred_val_base = rf_base.predict(X_val_scaled)
print("\n=== EVALUASI BASE MODEL (VALIDATION SET) ===")
print(classification_report(y_val, y_pred_val_base))

Training Base Model Random Forest...

=== EVALUASI BASE MODEL (VALIDATION SET) ===
              precision    recall  f1-score   support

           0       0.89      0.79      0.84       332
           1       0.77      0.84      0.80       332
           2       0.83      0.85      0.84       331

    accuracy                           0.83       995
   macro avg       0.83      0.83      0.83       995
weighted avg       0.83      0.83      0.83       995



## 3. Eksekusi Hyperparameter Tuning

In [4]:
# HYPERPARAMETER TUNING DENGAN GRIDSEARCHCV
print("Mulai eksekusi Hyperparameter Tuning...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Proses pencarian parameter terbaik
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nParameter Terbaik: {grid_search.best_params_}")

Mulai eksekusi Hyperparameter Tuning...
Fitting 5 folds for each of 81 candidates, totalling 405 fits

Parameter Terbaik: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


## 4. Pengujian Metrik Evaluasi Final

In [5]:
# EVALUASI MODEL TERBAIK KE TESTING SET
best_rf = grid_search.best_estimator_
y_pred_test = best_rf.predict(X_test_scaled)

print("=== EVALUASI MATRIX FINAL (TESTING SET) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_test, average='weighted'):.4f}\n")

print("Classification Report Detail:")
print(classification_report(y_test, y_pred_test))

=== EVALUASI MATRIX FINAL (TESTING SET) ===
Accuracy  : 0.8101
Precision : 0.8147
Recall    : 0.8101
F1-Score  : 0.8103

Classification Report Detail:
              precision    recall  f1-score   support

           0       0.89      0.77      0.83       331
           1       0.76      0.79      0.77       332
           2       0.80      0.87      0.83       332

    accuracy                           0.81       995
   macro avg       0.81      0.81      0.81       995
weighted avg       0.81      0.81      0.81       995

